In [4]:
from copy import deepcopy

import os
import sys
from pathlib import Path

os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import tqdm.auto
import tqdm.std

tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.auto.trange = tqdm.std.trange

import logging

import pandas as pd
import whisperx
from IPython.display import Audio, display

from drzavni_posao import (
    CharacterAssignerConfig,
    DiarizerConfig,
    EmbedderConfig,
    MediaStore,
    MlflowTracker,
    PipelineConfig,
    SR,
    Stage,
    Transcriber,
    TranscriberConfig,
    run_pipeline,
)
from drzavni_posao.comparison import (
    cast_accuracy,
    compare_all,
    load_transcripts,
    scribe_transcripts,
    text_of,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("mlflow.system_metrics").setLevel(logging.WARNING)

logger = logging.getLogger("drzavni_posao.pipeline")

from drzavni_posao.constants import CYRILLIC_TO_LATIN


/Users/luka.erceg/drzavni_posao/.venv/lib/python3.12/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 7: dlopen(/U

In [ ]:
EXPERIMENT_NAME = (
    "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
)
AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
TRANSCRIPT_PATH = Path(
    "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/transcripts"
)

HF_TOKEN = dbutils.secrets.get(scope="luka-erceg", key="hf_token")

SCRIBE_DIR = TRANSCRIPT_PATH / "elevenlabs"

DEVICE = "cuda"
COMPUTE_TYPE = "bfloat16"
EMBEDDER_MODEL = "pyannote/wespeaker-voxceleb-resnet34-LM"

ASR_MODELS = {
    "sr-cmb": "/Volumes/workspace/default/luka_erceg_models/whisper-sr-ct2",
    "large-v3": "large-v3",
    "large-v3-turbo": "large-v3-turbo",
}
ALIGN_MODELS = {
    "classla": "classla/wav2vec2-xls-r-parlaspeech-hr",
}

CAST_HOTWORDS = (
    "Đorđe Čvarkov, Dragan Torbica, Boškić, Dobrosavljev, "
    "Marijanović, Živan Radosavljević, Frau Šilovička"
)

NGRAM5 = {"no_repeat_ngram_size": 5, "repetition_penalty": 1.0}
DECODING_VARIANTS = {
    "baseline": {},
    "ngram3": {"no_repeat_ngram_size": 3, "repetition_penalty": 1.0},
    "ngram5": NGRAM5,
    "ngram5-numerals": {**NGRAM5, "suppress_numerals": True},
    "ngram5-hotwords": {**NGRAM5, "hotwords": CAST_HOTWORDS},
    "ngram5-beam10": {**NGRAM5, "beam_size": 10},
}

ALIGN_MODEL_KEY = "classla"

RUNS = [
    ("large-v3", "ngram5"),
    ("large-v3", "baseline"),
    ("large-v3", "ngram3"),
    ("large-v3", "ngram5-numerals"),
    ("large-v3", "ngram5-hotwords"),
    ("large-v3", "ngram5-beam10"),
]

EPISODES = set(scribe_transcripts(SCRIBE_DIR))

tracker = MlflowTracker(EXPERIMENT_NAME)


In [ ]:
# Episode 60 alignment hypothesis: source output and affected audio.
DIAGNOSTIC_SOURCE_RUN_ID = "b66150772bc043d091f7b4c9ff606ce0"
DIAGNOSTIC_EPISODE_KEY = "60"
DIAGNOSTIC_CLIPS = [(312.0, 322.0), (322.0, 332.0), (332.0, 342.0)]
DIAGNOSTIC_WINDOW = (DIAGNOSTIC_CLIPS[0][0], DIAGNOSTIC_CLIPS[-1][1])

diagnostic_config = tracker._configuration(DIAGNOSTIC_SOURCE_RUN_ID)
diagnostic_media = MediaStore(
    AUDIO_PATHS,
    Path("/tmp"),
    **diagnostic_config["audio"],
)
diagnostic_episode = diagnostic_media.episode(DIAGNOSTIC_EPISODE_KEY)
diagnostic_source = tracker.episode_run_states(
    DIAGNOSTIC_SOURCE_RUN_ID
)[DIAGNOSTIC_EPISODE_KEY]
tracker.restore_stage(
    diagnostic_episode,
    diagnostic_source.run_id,
    Stage.TRANSCRIPTION,
)
diagnostic_audio = diagnostic_episode.audio()
diagnostic_source_segments = [
    deepcopy(segment)
    for segment in diagnostic_episode.transcript.segments
    if segment["end"] > DIAGNOSTIC_WINDOW[0]
    and segment["start"] < DIAGNOSTIC_WINDOW[1]
]

print("Source-run text:")
print(" ".join(segment["text"].strip() for segment in diagnostic_source_segments))
display(
    Audio(
        diagnostic_audio[
            int(DIAGNOSTIC_WINDOW[0] * SR) : int(DIAGNOSTIC_WINDOW[1] * SR)
        ],
        rate=SR,
    )
)


In [ ]:
# Run the exact source ASR configuration on the full episode, without alignment.
diagnostic_transcriber = Transcriber(
    TranscriberConfig(**diagnostic_config["transcription"])
)
diagnostic_full_raw = diagnostic_transcriber.asr.transcribe(
    diagnostic_audio,
    batch_size=diagnostic_transcriber.config.batch_size,
)
for segment in diagnostic_full_raw["segments"]:
    segment["text"] = segment["text"].translate(CYRILLIC_TO_LATIN)

diagnostic_full_raw_segments = [
    segment
    for segment in diagnostic_full_raw["segments"]
    if segment["end"] > DIAGNOSTIC_WINDOW[0]
    and segment["start"] < DIAGNOSTIC_WINDOW[1]
]
assert diagnostic_full_raw_segments

display(
    pd.DataFrame(
        [
            {
                "start": segment["start"],
                "end": segment["end"],
                "avg_logprob": segment.get("avg_logprob"),
                "text": segment["text"].strip(),
            }
            for segment in diagnostic_full_raw_segments
        ]
    )
)

In [ ]:
# Re-run only the bad 30-second region as three independent 10-second clips.
def shift_timestamps(segments, offset):
    for segment in segments:
        for key in ("start", "end"):
            if key in segment:
                segment[key] += offset
        for word in segment.get("words", []):
            for key in ("start", "end"):
                if key in word:
                    word[key] += offset


def transcribe_clip(transcriber, start, end):
    clip = diagnostic_audio[int(start * SR) : int(end * SR)]
    raw = transcriber.asr.transcribe(
        clip,
        batch_size=transcriber.config.batch_size,
    )
    for segment in raw["segments"]:
        segment["text"] = segment["text"].translate(CYRILLIC_TO_LATIN)

    aligned = whisperx.align(
        deepcopy(raw["segments"]),
        transcriber.align_model,
        transcriber.align_meta,
        clip,
        transcriber.config.device,
        return_char_alignments=False,
    )
    raw_segments = deepcopy(raw["segments"])
    aligned_segments = aligned["segments"]
    shift_timestamps(raw_segments, start)
    shift_timestamps(aligned_segments, start)
    return {"clip": (start, end), "raw": raw_segments, "aligned": aligned_segments}


try:
    diagnostic_retries = [
        transcribe_clip(diagnostic_transcriber, start, end)
        for start, end in DIAGNOSTIC_CLIPS
    ]
finally:
    diagnostic_transcriber.free()

print(f"Finished {len(diagnostic_retries)} short-window retries.")

In [ ]:
# Compare transcript text and the long-word failure signal.
diagnostic_retry_segments = [
    segment
    for retry in diagnostic_retries
    for segment in retry["aligned"]
]

for retry in diagnostic_retries:
    start, end = retry["clip"]
    print(f"\nRaw ASR {start:.0f}–{end:.0f}s:")
    print(" ".join(segment["text"].strip() for segment in retry["raw"]))


def word_rows(mode, segments):
    return [
        {
            "mode": mode,
            "start": word["start"],
            "end": word["end"],
            "duration": word["end"] - word["start"],
            "score": word.get("score"),
            "word": word["word"],
            "segment": segment["text"].strip(),
        }
        for segment in segments
        for word in segment.get("words", [])
        if "start" in word and "end" in word
    ]


diagnostic_words = pd.DataFrame(
    word_rows("source", diagnostic_source_segments)
    + word_rows("short retry", diagnostic_retry_segments)
)
assert len(diagnostic_retries) == len(DIAGNOSTIC_CLIPS)
assert not diagnostic_words.empty

diagnostic_summary = pd.DataFrame(
    [
        {
            "mode": mode,
            "words": len(group),
            "max_word_seconds": group["duration"].max(),
            "words_over_2s": (group["duration"] > 2).sum(),
            "mean_alignment_score": group["score"].mean(),
        }
        for mode, group in diagnostic_words.groupby("mode")
    ]
).set_index("mode")

display(diagnostic_summary)
display(
    diagnostic_words[diagnostic_words["duration"] >= 1.5]
    .sort_values(["mode", "start"])
    .reset_index(drop=True)
)
print(
    "Hypothesis supported if the short-window raw text contains audible words "
    "missing from the source text and its long-word spans shrink."
)

In [ ]:
# Find source-run episodes at least as low-confidence as episode 60.
import json
from pathlib import Path
from tempfile import TemporaryDirectory

parent = tracker.client.get_run(DIAGNOSTIC_SOURCE_RUN_ID)
episode_runs = tracker.client.search_runs(
    [parent.info.experiment_id],
    filter_string=(
        f"tags.mlflow.parentRunId = '{DIAGNOSTIC_SOURCE_RUN_ID}' "
        "AND tags.run_level = 'episode'"
    ),
    max_results=1000,
)

confidence = pd.DataFrame(
    [
        {
            "episode": run.data.tags["episode_key"],
            "run_id": run.info.run_id,
            "p10": run.data.metrics.get("transcription.alignment_score_p10"),
            "median": run.data.metrics.get("transcription.alignment_score_median"),
        }
        for run in episode_runs
        if "episode_key" in run.data.tags
    ]
).dropna()
episode_60 = confidence[confidence["episode"] == DIAGNOSTIC_EPISODE_KEY].iloc[0]
candidates = confidence[
    (confidence["p10"] <= episode_60["p10"])
    | (confidence["median"] <= episode_60["median"])
].copy()

stretch = []
for run_id in candidates["run_id"]:
    with TemporaryDirectory() as directory:
        path = tracker.client.download_artifacts(
            run_id,
            "transcription/transcript.json",
            directory,
        )
        transcript = json.loads(Path(path).read_text(encoding="utf-8"))
    durations = [
        word["end"] - word["start"]
        for segment in transcript["segments"]
        for word in segment.get("words", [])
        if "start" in word and "end" in word
    ]
    stretch.append(
        {
            "max_word_seconds": max(durations, default=0),
            "words_over_2s": sum(duration > 2 for duration in durations),
        }
    )

candidates[["max_word_seconds", "words_over_2s"]] = pd.DataFrame(
    stretch,
    index=candidates.index,
)
print(
    f"{len(candidates)} of {len(confidence)} episodes are at least as low-confidence "
    "as episode 60 on p10 or median alignment score."
)
display(
    candidates.sort_values(["median", "p10"])[
        ["episode", "p10", "median", "max_word_seconds", "words_over_2s", "run_id"]
    ].reset_index(drop=True)
)

## Hard-word candidate labeling

Mine likely misspellings from disagreements between the two full exports and malformed cast names. Alternate context is matched by timestamp. Candidates with alignment score below 0.4 or local context agreement below 0.5 are excluded by default, so alignment failures do not enter the spelling benchmark.

Review each audio clip, enter the correct spelling, and save accepted labels to `data/hard_words.jsonl`. Skipped candidates are remembered separately.

In [1]:
# Mine or load review candidates from the two full local exports.
import json
import random
from pathlib import Path

from tqdm.auto import tqdm

from drzavni_posao import (
    HardWordExample,
    MediaStore,
    load_word_benchmark,
    load_word_candidates,
    mine_word_candidates,
    save_word_benchmark,
    save_word_candidates,
)

PRIMARY_RUN_ID = "1c519fba673841fa9188addf9a720ae6"
COMPARISON_RUN_ID = "4bd7a1a8d9024fd08c00c0e238bcc6b8"
CANDIDATE_LIMIT = 1000
CANDIDATE_VIEW = "non_cast"  # "all", "cast", or "non_cast"
CANDIDATE_ORDER_SEED = 42
REMINE_CANDIDATES = False

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PRIMARY_EXPORT = REPO_ROOT / "data" / "mlflow_exports" / PRIMARY_RUN_ID
COMPARISON_EXPORT = REPO_ROOT / "data" / "mlflow_exports" / COMPARISON_RUN_ID
AUDIO_PATHS = REPO_ROOT / "downloads"
CANDIDATE_PATH = REPO_ROOT / "data" / "hard_word_candidates_v4.jsonl"
WORD_LABEL_PATH = REPO_ROOT / "data" / "hard_words.jsonl"
REVIEW_PATH = REPO_ROOT / "data" / "hard_word_review.json"

primary_config = json.loads((PRIMARY_EXPORT / "config.json").read_text(encoding="utf-8"))
media = MediaStore(
    AUDIO_PATHS,
    REPO_ROOT / "data",
    catalog_path=REPO_ROOT / "data" / "youtube_episodes.json",
    **primary_config["audio"],
)
available_episode_keys = {episode.key for episode in media.episodes()}

if REMINE_CANDIDATES or not CANDIDATE_PATH.exists():
    progress_bar = tqdm(total=len(available_episode_keys), desc="Mining candidates")

    def update_mining_progress(done, total):
        progress_bar.total = total
        progress_bar.update(done - progress_bar.n)

    candidates = mine_word_candidates(
        PRIMARY_EXPORT,
        COMPARISON_EXPORT,
        episode_keys=available_episode_keys,
        max_candidates=CANDIDATE_LIMIT,
        progress=update_mining_progress,
    )
    progress_bar.close()
    save_word_candidates(CANDIDATE_PATH, candidates)
else:
    candidates = load_word_candidates(CANDIDATE_PATH)

if CANDIDATE_VIEW == "cast":
    candidates = [candidate for candidate in candidates if "cast_spelling" in candidate.reasons]
elif CANDIDATE_VIEW == "non_cast":
    candidates = [candidate for candidate in candidates if "cast_spelling" not in candidate.reasons]
elif CANDIDATE_VIEW != "all":
    raise ValueError(f"Unknown CANDIDATE_VIEW: {CANDIDATE_VIEW}")

# Preserve priority within each episode while alternating episodes between reviews.
original_candidate_ids = [candidate.id for candidate in candidates]
candidates_by_episode = {}
for candidate in candidates:
    candidates_by_episode.setdefault(candidate.episode_key, []).append(candidate)
episode_order = list(candidates_by_episode)
random.Random(CANDIDATE_ORDER_SEED).shuffle(episode_order)
candidates = [
    candidate
    for candidate_index in range(
        max(map(len, candidates_by_episode.values()), default=0)
    )
    for episode_key in episode_order
    for candidate in candidates_by_episode[episode_key][candidate_index : candidate_index + 1]
]
if sorted(candidate.id for candidate in candidates) != sorted(original_candidate_ids):
    raise RuntimeError("Candidate episode shuffle changed the candidate set")

labels = {
    example.id: example
    for example in (
        load_word_benchmark(WORD_LABEL_PATH)
        if WORD_LABEL_PATH.exists() and WORD_LABEL_PATH.stat().st_size
        else []
    )
}
review_state = (
    json.loads(REVIEW_PATH.read_text(encoding="utf-8"))
    if REVIEW_PATH.exists()
    else {"reviewed": []}
)
reviewed_ids = set(review_state.get("reviewed", [])) | set(labels)
print(
    f"{len(candidates)} candidates, {len(labels)} accepted labels, "
    f"{len(reviewed_ids)} reviewed"
)

767 candidates, 15 accepted labels, 15 reviewed


In [ ]:
# Interactive hard-word reviewer. Saves after every Accept or Skip.
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

from drzavni_posao.constants import SR

AUDIO_CONTEXT_SECONDS = 3.0
CATEGORY_OPTIONS = [
    "name",
    "dialect",
    "diacritic",
    "inflection",
    "word_boundary",
    "ordinary",
    "other",
    "alignment",
]
candidate_ids = {candidate.id for candidate in candidates}
position = next(
    (index for index, candidate in enumerate(candidates) if candidate.id not in reviewed_ids),
    0,
)
loaded_episode_key = None
loaded_audio = None

output = widgets.Output()
clip_output = widgets.Output()
status = widgets.HTML()
progress = widgets.IntProgress(
    min=0,
    max=len(candidates),
    description="Reviewed",
    bar_style="success",
)
expected_input = widgets.Text(
    description="Correct text",
    placeholder="Type the correct word or short phrase",
    layout=widgets.Layout(width="70%"),
)
category_choice = widgets.Dropdown(
    options=CATEGORY_OPTIONS,
    description="Category",
)
previous_button = widgets.Button(description="Previous", icon="arrow-left")
replay_button = widgets.Button(description="Replay", icon="refresh")
next_button = widgets.Button(description="Next", icon="arrow-right")
skip_button = widgets.Button(description="Skip", button_style="warning")
save_button = widgets.Button(description="Accept & Next", button_style="success", icon="save")


def persist_word_review():
    ordered_labels = sorted(
        labels.values(),
        key=lambda item: (item.episode_key, item.start, item.end, item.id),
    )
    save_word_benchmark(WORD_LABEL_PATH, ordered_labels)
    REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    REVIEW_PATH.write_text(
        json.dumps({"reviewed": sorted(reviewed_ids)}, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def current_candidate():
    return candidates[position]


def load_candidate_audio(candidate):
    global loaded_episode_key, loaded_audio
    if loaded_episode_key != candidate.episode_key:
        if loaded_episode_key is not None:
            media.episode(loaded_episode_key).clear_audio()
        loaded_episode_key = candidate.episode_key
        loaded_audio = media.episode(candidate.episode_key).audio()
    return loaded_audio


def play_candidate(autoplay=True):
    candidate = current_candidate()
    with clip_output:
        clear_output(wait=True)
        try:
            audio = load_candidate_audio(candidate)
        except Exception as error:
            print(f"Could not load episode {candidate.episode_key}: {error}")
            return
        trim_start = primary_config["audio"]["trim_start"]
        start = max(0.0, candidate.start - trim_start - AUDIO_CONTEXT_SECONDS)
        end = min(
            len(audio) / SR,
            candidate.end - trim_start + AUDIO_CONTEXT_SECONDS,
        )
        display(
            Audio(
                audio[int(start * SR) : int(end * SR)].copy(),
                rate=SR,
                autoplay=autoplay,
            )
        )


def reviewed_count():
    return len(candidate_ids & reviewed_ids)


def show_candidate():
    if not candidates:
        with output:
            clear_output(wait=True)
            print("No candidates were mined.")
        for control in [
            expected_input,
            category_choice,
            previous_button,
            replay_button,
            next_button,
            skip_button,
            save_button,
        ]:
            control.disabled = True
        return

    candidate = current_candidate()
    saved = labels.get(candidate.id)
    expected_input.value = saved.expected if saved else ""
    category_choice.value = (
        saved.category
        if saved and saved.category in CATEGORY_OPTIONS
        else candidate.suggested_category
    )
    progress.value = reviewed_count()

    with output:
        clear_output(wait=True)
        print(
            f"{position + 1}/{len(candidates)}  reviewed {reviewed_count()}/{len(candidates)}  "
            f"episode {candidate.episode_key}  {candidate.start:.2f}–{candidate.end:.2f}s raw"
        )
        print(f"1c51: {candidate.primary}")
        print(f"4bd7: {candidate.alternate or '[missing]'}")
        print(f"1c51 context: {candidate.primary_context}")
        print(f"4bd7 context: {candidate.alternate_context}")
        context_agreement = (
            "n/a"
            if candidate.context_agreement is None
            else f"{candidate.context_agreement:.2f}"
        )
        print(
            f"reasons={', '.join(candidate.reasons)}  frequency={candidate.frequency}  "
            f"align={candidate.alignment_score}  "
            f"context_agreement={context_agreement}  "
            f"priority={candidate.priority:.1f}"
        )

    previous_button.disabled = position == 0
    next_button.disabled = position == len(candidates) - 1
    status.value = (
        f"<b>Saved: {saved.expected}</b>"
        if saved
        else "<span style='color:#777'>Not accepted</span>"
    )
    play_candidate()


def step_candidate(delta):
    global position
    position = max(0, min(len(candidates) - 1, position + delta))
    show_candidate()


def advance_to_unreviewed():
    global position
    next_position = next(
        (
            index
            for index in range(position + 1, len(candidates))
            if candidates[index].id not in reviewed_ids
        ),
        None,
    )
    if next_position is None:
        next_position = next(
            (
                index
                for index, candidate in enumerate(candidates)
                if candidate.id not in reviewed_ids
            ),
            min(position + 1, len(candidates) - 1),
        )
    position = next_position
    show_candidate()


def save_current_word(_):
    candidate = current_candidate()
    expected = expected_input.value.strip()
    if not expected:
        status.value = "<b style='color:#b00'>Enter the correct word or phrase.</b>"
        return
    labels[candidate.id] = HardWordExample(
        id=candidate.id,
        episode_key=candidate.episode_key,
        start=candidate.start,
        end=candidate.end,
        expected=expected,
        category=category_choice.value,
        before=candidate.before,
        after=candidate.after,
    )
    reviewed_ids.add(candidate.id)
    persist_word_review()
    advance_to_unreviewed()


def skip_current_word(_):
    candidate = current_candidate()
    labels.pop(candidate.id, None)
    reviewed_ids.add(candidate.id)
    persist_word_review()
    advance_to_unreviewed()


previous_button.on_click(lambda _: step_candidate(-1))
replay_button.on_click(lambda _: play_candidate())
next_button.on_click(lambda _: step_candidate(1))
skip_button.on_click(skip_current_word)
save_button.on_click(save_current_word)
display(
    widgets.VBox(
        [
            progress,
            output,
            clip_output,
            expected_input,
            category_choice,
            status,
            widgets.HBox(
                [previous_button, replay_button, next_button, skip_button, save_button]
            ),
        ]
    )
)
show_candidate()